## Imports

In [1]:
import os
import pandas as pd
from pathlib import Path
import pydicom
import pydicom_seg as dcmseg
from radiomics import featureextractor
import numpy as np
import SimpleITK as sitk
#from utils_pyradiomics import create_path_df
import matplotlib.pyplot as plt

In [2]:
full_dataset = pd.read_csv(os.path.expanduser('~/project/xAI-in-NSCLC/FULL_radiomics_features_per_slice.csv'))
other_test_dataset = pd.read_csv(os.path.expanduser('~/project/xAI-in-NSCLC/TEST1_FULL_radiomics_features_per_slice.csv'))

In [4]:

print(full_dataset.shape)
print(other_test_dataset.shape)
display(full_dataset.head(5))
display(other_test_dataset.head(5))

(7223, 662)
(124, 662)


,PatientID,slice_no,original_shape2D_Elongation,original_shape2D_MajorAxisLength,original_shape2D_MaximumDiameter,original_shape2D_MeshSurface,original_shape2D_MinorAxisLength,original_shape2D_Perimeter,original_shape2D_PerimeterSurfaceRatio,original_shape2D_PixelSurface,...,wavelet-L_gldm_LargeDependenceLowGrayLevelEmphasis,wavelet-L_gldm_LowGrayLevelEmphasis,wavelet-L_gldm_SmallDependenceEmphasis,wavelet-L_gldm_SmallDependenceHighGrayLevelEmphasis,wavelet-L_gldm_SmallDependenceLowGrayLevelEmphasis,wavelet-L_ngtdm_Busyness,wavelet-L_ngtdm_Coarseness,wavelet-L_ngtdm_Complexity,wavelet-L_ngtdm_Contrast,wavelet-L_ngtdm_Strength
0,LUNG1-001,65,0.544937,13.659473,13.566840,78.678131,7.443553,35.768962,0.454624,79.154968,...,0.012320,0.012319,0.981928,121004.765060,0.012319,0.001196,0.045945,232302.606827,188.887364,4646.827408
1,LUNG1-001,66,0.513136,25.130245,25.633603,248.432159,12.895239,66.822663,0.268978,248.908997,...,0.004195,0.003963,0.925287,146055.210728,0.003905,0.000540,0.035563,247872.636083,20.744195,2395.297089
2,LUNG1-001,67,0.455774,51.188040,50.706074,826.358795,23.330172,137.649725,0.166574,826.835632,...,0.001178,0.001177,0.948802,117398.479239,0.001176,0.000529,0.017956,322249.286049,2.527396,1310.173794
3,LUNG1-001,68,0.661752,64.091619,68.589178,2030.849457,42.412765,197.330095,0.097166,2031.326294,...,0.000826,0.000791,0.963954,131113.500000,0.000782,0.000399,0.012122,963479.546047,1.742460,1705.476464
4,LUNG1-001,69,0.606515,76.090875,81.967283,2584.934235,46.150227,232.290046,0.089863,2585.411072,...,0.000385,0.000384,0.953871,184226.635620,0.000383,0.000569,0.006616,960106.575390,1.332179,1232.922317


,PatientID,slice_no,original_shape2D_Elongation,original_shape2D_MajorAxisLength,original_shape2D_MaximumDiameter,original_shape2D_MeshSurface,original_shape2D_MinorAxisLength,original_shape2D_Perimeter,original_shape2D_PerimeterSurfaceRatio,original_shape2D_PixelSurface,...,wavelet-L_gldm_LargeDependenceLowGrayLevelEmphasis,wavelet-L_gldm_LowGrayLevelEmphasis,wavelet-L_gldm_SmallDependenceEmphasis,wavelet-L_gldm_SmallDependenceHighGrayLevelEmphasis,wavelet-L_gldm_SmallDependenceLowGrayLevelEmphasis,wavelet-L_ngtdm_Busyness,wavelet-L_ngtdm_Coarseness,wavelet-L_ngtdm_Complexity,wavelet-L_ngtdm_Contrast,wavelet-L_ngtdm_Strength
0,LUNG1-001,65,0.544937,13.659473,13.566840,78.678131,7.443553,35.768962,0.454624,79.154968,...,0.012280,0.012280,1.000000,125063.903614,0.012280,0.001803,0.032302,330246.365462,259.038855,3318.060074
1,LUNG1-001,66,0.513136,25.130245,25.633603,248.432159,12.895239,66.822663,0.268978,248.908997,...,0.004028,0.004027,0.959770,148052.473180,0.004027,0.000760,0.027314,270060.682857,26.962083,2043.837332
2,LUNG1-001,67,0.455774,51.188040,50.706074,826.358795,23.330172,137.649725,0.166574,826.835632,...,0.001181,0.001179,0.941881,119674.300205,0.001179,0.000564,0.016386,349631.223548,2.663807,1260.683472
3,LUNG1-001,68,0.661752,64.091619,68.589178,2030.849457,42.412765,197.330095,0.097166,2031.326294,...,0.000724,0.000722,0.962546,129524.505112,0.000722,0.000347,0.014009,828867.395773,1.463536,2027.830170
4,LUNG1-001,69,0.606515,76.090875,81.967283,2584.934235,46.150227,232.290046,0.089863,2585.411072,...,0.000386,0.000384,0.939096,176584.865384,0.000384,0.000411,0.009037,686995.717716,0.871685,1925.246644


## Loading dicom files

In [ ]:
def read_scans(ct_path, seg_path):

    seg = pydicom.dcmread(list(seg_path.glob('*.dcm'))[0])
    result_seg = seg_reader.read(seg)
    dcm_paths = sorted(ct_path.glob('*.dcm'))
    dcm_files = ser_reader.GetGDCMSeriesFileNames(str(ct_path))
    ser_reader.SetFileNames(dcm_files)
    dcms = ser_reader.Execute()

    return dcms, result_seg, seg

def get_neoplasm_segment_image(result_seg):
    for seg_num, info in result_seg.segment_infos.items():
        if 'Neoplasm' in info.get('SegmentLabel', ''):
            return result_seg.segment_image(seg_num)
    raise ValueError('Neoplasm segment not found')

In [ ]:
def create_path_df(general_dir):
    
    path_records = []

    for patient_dir in general_dir.iterdir():
        if not patient_dir.is_dir():
            continue

        scan_id = patient_dir.name
        
        for study_dir in patient_dir.iterdir():
            if not study_dir.is_dir():
                continue

            for series_dir in study_dir.iterdir():
                if not series_dir.is_dir():
                    continue

                #select whether ct scan series or segmentation based on name/length
                if 'Segmentation' in series_dir.name and any(series_dir.glob('*.dcm')):
                    seg_series = series_dir
                    continue

                if any(series_dir.glob('*.dcm')) and len(list(series_dir.glob('*.dcm'))) >= 10:
                    ct_series = series_dir

            if ct_series is not None and seg_series is not None:
                path_records.append({
                    'scan_id': scan_id,
                    'path_ct': ct_series,
                    'path_mask':seg_series
                })
            else:
                print(f"No valid paths for {patient_dir.name}/{study_dir.name}: ct_series={ct_series is not None}, seg_series={seg_series is not None}")

    path_df = pd.DataFrame(path_records, columns=['scan_id', 'path_ct', 'path_mask'])

    return path_df

In [ ]:
general_dir = Path(os.path.expanduser('~/project/xAI-in-NSCLC/NSCLC-Radiomics'))

path_df = create_path_df(general_dir)



In [ ]:
path_df = path_df[:5]

In [ ]:
#Edit this so that you have a toggle for fixing the segmentation

is_fix_segmentation = bool(True)

In [ ]:

def extract_slice(img, slice_no):
    size = list(img.GetSize())
    index = [0, 0, int(slice_no)]

    size[2] = 0  # extract 2D slice
    slice_img = sitk.Extract(img, size, index)

    # print(f"origin: {slice_img.GetOrigin()}")
    # print(f"size: {slice_img.GetSize()}")
    # print(f"spacing: {slice_img.GetSpacing()}")
    # print(f"direction: {slice_img.GetDirection()}\n")

    # # Copy only spacing and origin for X/Y
    # spacing = img.GetSpacing()
    # origin = img.GetOrigin()
    # direction = img.GetDirection()

    # # 2D spacing = first two components
    # slice_img.SetSpacing(spacing[:2])

    # # 2D origin = first two components
    # slice_img.SetOrigin(origin[:2])

    # # 2D direction = 2×2 top-left of 3×3 matrix
    # slice_img.SetDirection(direction[:4])

    return slice_img

In [ ]:
def extract_per_slice(extractor, fixed_seg, sitk_dcms, scan_id, records):
    for slice_no in range(sitk_dcms.GetSize()[2]):
        seg_slice = extract_slice(fixed_seg, slice_no)
        img_slice = extract_slice(sitk_dcms, slice_no)
        # print('Seg')
        # print(f"origin: {seg_slice.GetOrigin()}")
        # print(f"size: {seg_slice.GetSize()}")
        # print(f"spacing: {seg_slice.GetSpacing()}")
        # print(f"direction: {seg_slice.GetDirection()}\n")
        
        # print('\nimg')
        # print(f"origin: {img_slice.GetOrigin()}")
        # print(f"size: {img_slice.GetSize()}")
        # print(f"spacing: {img_slice.GetSpacing()}")
        # print(f"direction: {img_slice.GetDirection()}\n")
        
        # Check if segmentation contains label 1 -> some slices will have no segmentation (label 0)
        if 1 not in sitk.GetArrayViewFromImage(seg_slice):
            continue

        features = extractor.execute(img_slice, seg_slice, label=1)
        record = {'PatientID': scan_id,
                'slice_no': slice_no}
        
        record.update(features)
        records.append(record)
    return records

In [ ]:
def fix_seg(seg_img, ct_imgs):
    fixed_seg = sitk.Cast(seg_img, sitk.sitkUInt8)
    print(f"origin: {fixed_seg.GetOrigin()}")
    print(f"size: {fixed_seg.GetSize()}")
    print(f"spacing: {fixed_seg.GetSpacing()}")
    print(f"direction: {fixed_seg.GetDirection()}\n")
    
    return fixed_seg

In [ ]:

def initialize_feature_extractor():
    paramsFile = os.path.expanduser('~/project/xAI-in-NSCLC/CEM_extraction.yaml')
    extractor = featureextractor.RadiomicsFeatureExtractor(paramsFile, shape2D=True, force2D=True,
                                                            force2Ddimension=0, resampledPixelSpacing=None) #originally: force2DDimension=True, now set to 0 for axial plane
    extractor.addProvenance(False) #It's not necessary to resample PixelSpacing since it is consistent across the dataset.
    extractor.disableAllFeatures()
    extractor.enableImageTypes(Original={})

    extractor.enableFeatureClassByName('firstorder', enabled=True)
    extractor.enableFeatureClassByName('shape2D', enabled=True)
    extractor.enableFeatureClassByName('glcm', enabled=True)
    extractor.enableFeatureClassByName('glrlm', enabled=True)
    extractor.enableFeatureClassByName('glszm', enabled=True)
    extractor.enableFeatureClassByName('gldm', enabled=True)
    extractor.enableFeatureClassByName('ngtdm', enabled=True)
    return extractor

In [ ]:
import logging

# hiding pyradiomics info, clogs up terminal
logging.getLogger('radiomics').setLevel(logging.ERROR)
logging.getLogger('radiomics.featureextractor').setLevel(logging.ERROR)
logging.getLogger('pyradiomics').setLevel(logging.ERROR)

In [ ]:
seg_reader = dcmseg.SegmentReader()
ser_reader = sitk.ImageSeriesReader()
extractor = initialize_feature_extractor()

records = []

scan_data = {}
for _, row in path_df.iterrows():
    scan_id = row['scan_id']
    ct_path = row['path_ct']
    mask_path = row['path_mask']

    print(f'started processing scan: {scan_id}')

    seg = pydicom.dcmread(list(mask_path.glob('*.dcm'))[0])
    seg = seg_reader.read(seg)
    dcm_files = ser_reader.GetGDCMSeriesFileNames(str(ct_path))
    ser_reader.SetFileNames(dcm_files)
    sitk_dcms = ser_reader.Execute()

    # find neoplasm label within the segmentation and extract segment
    seg_infos = seg.segment_infos

    for seg_num, info in seg_infos.items():

        if 'Neoplasm' in info.get('SegmentLabel', ''):
            neo_seg_num = seg_num
            break
        else:
            continue
    print("Segment number chosen:", neo_seg_num)
    print("Segment label:", seg_infos[neo_seg_num]['SegmentLabel']) 
    neoplasm_segment = seg.segment_image(neo_seg_num)
                
        #Spatial alignment needs to be fixed due to differences in datahandling between dcmseg and sitk
        #pyradiomics will throw error if it thinks it is even 0.001mm off
        
    fixed_seg = fix_seg(neoplasm_segment, sitk_dcms)
                
    if fixed_seg.GetSize() != sitk_dcms.GetSize():
        print(f"Skipping {scan_id} due to size mismatch: seg={fixed_seg.GetSize()}, ct={sitk_dcms.GetSize()}")
        mismatched_scans.append(scan_id)
        break
                
    fixed_seg.CopyInformation(sitk_dcms)

    records = extract_per_slice(extractor, fixed_seg, sitk_dcms, scan_id, records)

In [ ]:
records

In [ ]:
features = pd.DataFrame(records)
features.head(10)

In [ ]:
def fix_seg(seg_img, ct_imgs):
    fixed_seg = sitk.Cast(seg_img, sitk.sitkUInt8)
    return fixed_seg

In [ ]:
seg_reader = dcmseg.SegmentReader()
ser_reader = sitk.ImageSeriesReader()


scan_data = {}
for _, row in path_df.iterrows():
    scan_id = row['scan_id']
    ct_path = row['path_ct']
    mask_path = row['path_mask']

    sitk_dcms, result_seg, seg = read_scans(ct_path, mask_path)
    neoplasm_segment_img = get_neoplasm_segment_image(result_seg)

    if is_fix_segmentation == True:
        neoplasm_segment_img = fix_seg(neoplasm_segment_img, sitk_dcms)
        neoplasm_segment_img.CopyInformation(sitk_dcms)

    ct_array = sitk.GetArrayFromImage(sitk_dcms)
    seg_array = sitk.GetArrayFromImage(neoplasm_segment_img)
    slice_indices = list(np.where(seg_array.sum(axis=(1, 2)) > 0)[0])

    scan_data[scan_id] = {
        'ct_array': ct_array,
        'seg_array': seg_array,
        'slice_indices': slice_indices,
    }

In [ ]:
def show_overlay(scan_id, slice_idx, alpha=0.5):
    data = scan_data[scan_id]
    ct_array = data['ct_array']
    seg_array = data['seg_array']

    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Display CT scan
    ax.imshow(ct_array[slice_idx], cmap='gray')
    
    # Overlay segmentation with transparency
    ax.imshow(seg_array[slice_idx], cmap='Reds', alpha=alpha)
    ax.set_title(f'{scan_id} slice {slice_idx}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# Create widgets for scan and mismatched slice selection
scan_selector = widgets.Dropdown(
    options=list(scan_data.keys()),
    description='Scan:',
    value=list(scan_data.keys())[0],
)

initial_slices = scan_data[scan_selector.value]['slice_indices']
if not initial_slices:
    raise RuntimeError(f'No segmented slices found for scan {scan_selector.value}')

slice_selector = widgets.SelectionSlider(
    options=initial_slices,
    description='Slice:',
    value=initial_slices[0],
    continuous_update=False,
)

alpha_selector = widgets.FloatSlider(
    min=0.0,
    max=1.0,
    step=0.1,
    value=0.5,
    description='Alpha:',
)


def update_slice_options(change):
    new_scan = change['new']
    new_slices = scan_data[new_scan]['slice_indices']
    slice_selector.options = new_slices
    slice_selector.value = new_slices[0] if new_slices else None

scan_selector.observe(update_slice_options, names='value')


def show_overlay(scan_id, slice_idx, alpha=0.5):
    data = scan_data[scan_id]
    ct_array = data['ct_array']
    seg_array = data['seg_array']

    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Display CT scan
    ax.imshow(ct_array[slice_idx], cmap='gray')
    
    # Overlay segmentation with transparency
    ax.imshow(seg_array[slice_idx], cmap='Reds', alpha=alpha)
    ax.set_title(f'{scan_id} slice {slice_idx}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close()

output = widgets.interactive_output(
    show_overlay,
    {
        'scan_id': scan_selector,
        'slice_idx': slice_selector,
        'alpha': alpha_selector,
    }
)

display(widgets.VBox([
    scan_selector,
    slice_selector,
    alpha_selector,
    output,
]))